# 🧺 Market Basket Analysis
## 🛒 Product-to-Product Recommendation

---

# 📌 Objetivo

Este notebook tem como objetivo analisar relações entre produtos comprados em conjunto.

A proposta é identificar padrões de coocorrência entre itens, permitindo responder perguntas como:

> Se um usuário comprou o produto A, quais produtos B são mais prováveis de aparecer no mesmo pedido?

Essa abordagem será utilizada como base para um sistema de recomendação produto → produto, semelhante a estratégias como:

- "Quem comprou este produto também comprou..."
- "Produtos frequentemente comprados juntos"
- "Itens complementares no carrinho"

---

# 🎯 Estratégia

Serão analisados pares de produtos presentes nos mesmos pedidos, calculando métricas como:

- Frequência de coocorrência
- Probabilidade condicional `P(B | A)`
- Lift entre produtos
- Ranking de recomendações por produto

In [1]:
# =========================================
# 📚 IMPORTAÇÃO DAS BIBLIOTECAS
# =========================================

import gc
import warnings

import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from itertools import combinations

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [2]:
# =========================================
# 📂 CARREGAMENTO DOS DADOS
# =========================================

DATA_PATH = Path("../data/raw")

order_products_prior = pd.read_csv(
    DATA_PATH / "order_products__prior.csv"
)

products = pd.read_csv(
    DATA_PATH / "products.csv"
)

aisles = pd.read_csv(
    DATA_PATH / "aisles.csv"
)

departments = pd.read_csv(
    DATA_PATH / "departments.csv"
)

print("✅ Dados carregados com sucesso!")
print("order_products_prior:", order_products_prior.shape)
print("products:", products.shape)

✅ Dados carregados com sucesso!
order_products_prior: (32434489, 4)
products: (49688, 4)


In [3]:
# =========================================
# 🛒 PRODUTOS ENRIQUECIDOS
# =========================================

products_enriched = (
    products
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

products_enriched.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [4]:
# =========================================
# ⚙️ AMOSTRAGEM DOS PEDIDOS
# =========================================

N_SAMPLE_ORDERS = 50_000
RANDOM_STATE = 42

sample_order_ids = (
    order_products_prior["order_id"]
    .drop_duplicates()
    .sample(
        n=N_SAMPLE_ORDERS,
        random_state=RANDOM_STATE
    )
)

sample_order_products = order_products_prior[
    order_products_prior["order_id"].isin(sample_order_ids)
]

print("Shape da amostra:", sample_order_products.shape)

sample_order_products.head()

Shape da amostra: (505250, 4)


,order_id,product_id,add_to_cart_order,reordered
125,14,20392,1,1
126,14,27845,2,1
127,14,162,3,1
128,14,2452,4,1
129,14,8575,5,1


In [5]:
# =========================================
# 🧺 CESTAS DE PRODUTOS
# =========================================

baskets = (
    sample_order_products
    .groupby("order_id")["product_id"]
    .apply(lambda x: list(set(x)))
)

print("Quantidade de cestas:", len(baskets))

baskets.head()

Quantidade de cestas: 50000


order_id
14     [162, 41890, 20995, 27845, 20392, 45066, 10096...
42                                 [33000, 13176, 19887]
97     [30850, 34565, 10761, 29963, 4366, 33439, 3318...
231    [14464, 32265, 40203, 28569, 14875, 29471, 212...
291    [19213, 3090, 24852, 39732, 16150, 7350, 3957,...
Name: product_id, dtype: object

In [6]:
# =========================================
# 🔗 CONTAGEM DE PARES DE PRODUTOS
# =========================================

pair_counter = Counter()

for product_list in baskets:
    if len(product_list) > 1:
        for pair in combinations(sorted(product_list), 2):
            pair_counter[pair] += 1

top_pairs = pair_counter.most_common(20)

top_pairs_df = pd.DataFrame(
    top_pairs,
    columns=["product_pair", "cooccurrence_count"]
)

top_pairs_df.head()

,product_pair,cooccurrence_count
0,"(13176, 47209)",991
1,"(13176, 21137)",924
2,"(21137, 24852)",841
3,"(24852, 47766)",788
4,"(21903, 24852)",787


# 🔍 Interpretação dos Pares

Após identificar os pares mais frequentes, os IDs dos produtos serão substituídos por seus respectivos nomes para facilitar a interpretação dos resultados.

In [7]:
# =========================================
# 🔍 NOMES DOS PRODUTOS
# =========================================

product_name_map = dict(
    zip(
        products_enriched["product_id"],
        products_enriched["product_name"]
    )
)

top_pairs_df["product_1_id"] = top_pairs_df["product_pair"].apply(lambda x: x[0])
top_pairs_df["product_2_id"] = top_pairs_df["product_pair"].apply(lambda x: x[1])

top_pairs_df["product_1"] = top_pairs_df["product_1_id"].map(product_name_map)
top_pairs_df["product_2"] = top_pairs_df["product_2_id"].map(product_name_map)

top_pairs_df[
    [
        "product_1",
        "product_2",
        "cooccurrence_count"
    ]
]

,product_1,product_2,cooccurrence_count
0,Bag of Organic Bananas,Organic Hass Avocado,991
1,Bag of Organic Bananas,Organic Strawberries,924
2,Organic Strawberries,Banana,841
3,Banana,Organic Avocado,788
4,Organic Baby Spinach,Banana,787
5,Bag of Organic Bananas,Organic Baby Spinach,740
6,Organic Strawberries,Organic Hass Avocado,656
7,Bag of Organic Bananas,Organic Raspberries,656
8,Strawberries,Banana,647
9,Banana,Large Lemon,613


In [8]:
# =========================================
# 📊 CÁLCULO OTIMIZADO DE CONFIDENCE E LIFT
# =========================================

gc.collect()

MIN_PAIR_COUNT = 20

total_orders_sample = sample_order_products["order_id"].nunique()

product_order_count_map = (
    sample_order_products
    .drop_duplicates(["order_id", "product_id"])
    ["product_id"]
    .value_counts()
    .to_dict()
)

pairs_df = pd.DataFrame(
    pair_counter.items(),
    columns=["product_pair", "cooccurrence_count"]
)

pairs_df["product_1_id"] = pairs_df["product_pair"].apply(lambda x: x[0])
pairs_df["product_2_id"] = pairs_df["product_pair"].apply(lambda x: x[1])

pairs_df = pairs_df.drop(columns=["product_pair"])

pairs_df = pairs_df[
    pairs_df["cooccurrence_count"] >= MIN_PAIR_COUNT
].copy()

pairs_df["product_1_id"] = pairs_df["product_1_id"].astype("int32")
pairs_df["product_2_id"] = pairs_df["product_2_id"].astype("int32")
pairs_df["cooccurrence_count"] = pairs_df["cooccurrence_count"].astype("int32")

pairs_ab = pairs_df.rename(
    columns={
        "product_1_id": "product_a_id",
        "product_2_id": "product_b_id"
    }
)

pairs_ba = pairs_df.rename(
    columns={
        "product_1_id": "product_b_id",
        "product_2_id": "product_a_id"
    }
)

directional_pairs = pd.concat(
    [pairs_ab, pairs_ba],
    ignore_index=True
)

directional_pairs["product_a_count"] = (
    directional_pairs["product_a_id"]
    .map(product_order_count_map)
)

directional_pairs["product_b_count"] = (
    directional_pairs["product_b_id"]
    .map(product_order_count_map)
)

directional_pairs["confidence"] = (
    directional_pairs["cooccurrence_count"] /
    directional_pairs["product_a_count"]
)

directional_pairs["product_b_support"] = (
    directional_pairs["product_b_count"] /
    total_orders_sample
)

directional_pairs["lift"] = (
    directional_pairs["confidence"] /
    directional_pairs["product_b_support"]
)

directional_pairs["confidence"] = directional_pairs["confidence"].astype("float32")
directional_pairs["product_b_support"] = directional_pairs["product_b_support"].astype("float32")
directional_pairs["lift"] = directional_pairs["lift"].astype("float32")

print("Quantidade de regras:", directional_pairs.shape)

directional_pairs.head()

Quantidade de regras: (17374, 8)


,cooccurrence_count,product_a_id,product_b_id,product_a_count,product_b_count,confidence,product_b_support,lift
0,20,2452,27845,156,2146,0.128205,0.04292,2.987072
1,41,20995,27845,521,2146,0.078695,0.04292,1.833523
2,59,27845,39475,2146,416,0.027493,0.00832,3.304448
3,89,27845,45066,2146,1214,0.041473,0.02428,1.708093
4,88,13176,33000,6001,515,0.014664,0.01030,1.423711


In [9]:
# =========================================
# 🏷️ NOMES DOS PRODUTOS
# =========================================

directional_pairs["product_a"] = (
    directional_pairs["product_a_id"]
    .map(product_name_map)
)

directional_pairs["product_b"] = (
    directional_pairs["product_b_id"]
    .map(product_name_map)
)

recommendation_rules = directional_pairs[
    [
        "product_a_id",
        "product_a",
        "product_b_id",
        "product_b",
        "cooccurrence_count",
        "confidence",
        "lift"
    ]
].copy()

recommendation_rules.head()

,product_a_id,product_a,product_b_id,product_b,cooccurrence_count,confidence,lift
0,2452,Naturals Chicken Nuggets,27845,Organic Whole Milk,20,0.128205,2.987072
1,20995,Organic Broccoli Florets,27845,Organic Whole Milk,41,0.078695,1.833523
2,27845,Organic Whole Milk,39475,Total Greek Strained Yogurt,59,0.027493,3.304448
3,27845,Organic Whole Milk,45066,Honeycrisp Apple,89,0.041473,1.708093
4,13176,Bag of Organic Bananas,33000,Pure Irish Butter,88,0.014664,1.423711


In [10]:
# =========================================
# ⭐ SCORE DE RECOMENDAÇÃO
# =========================================

recommendation_rules["recommendation_score"] = (
    recommendation_rules["lift"] *
    np.log1p(recommendation_rules["cooccurrence_count"]) *
    recommendation_rules["confidence"]
)

recommendation_rules.head()

,product_a_id,product_a,product_b_id,product_b,cooccurrence_count,confidence,lift,recommendation_score
0,2452,Naturals Chicken Nuggets,27845,Organic Whole Milk,20,0.128205,2.987072,1.165924
1,20995,Organic Broccoli Florets,27845,Organic Whole Milk,41,0.078695,1.833523,0.539304
2,27845,Organic Whole Milk,39475,Total Greek Strained Yogurt,59,0.027493,3.304448,0.371968
3,27845,Organic Whole Milk,45066,Honeycrisp Apple,89,0.041473,1.708093,0.318762
4,13176,Bag of Organic Bananas,33000,Pure Irish Butter,88,0.014664,1.423711,0.093712


In [11]:
# =========================================
# 🛒 FUNÇÃO DE RECOMENDAÇÃO
# =========================================

def recommend_products(
    product_name: str,
    top_n: int = 10,
    min_cooccurrence: int = 50,
    min_lift: float = 1.0,
    min_confidence: float = 0.005,
    sort_by: str = "recommendation_score"
) -> pd.DataFrame:
    
    product_name_clean = product_name.lower().strip()

    # Busca correspondência exata primeiro
    exact_match = products_enriched[
        products_enriched["product_name"]
        .str.lower()
        .str.strip()
        .eq(product_name_clean)
    ]

    # Se não encontrar exato, usa busca parcial
    if not exact_match.empty:
        matched_product = exact_match.iloc[0]
    else:
        partial_match = products_enriched[
            products_enriched["product_name"]
            .str.lower()
            .str.contains(product_name_clean, regex=False)
        ]

        if partial_match.empty:
            print("Produto não encontrado.")
            return pd.DataFrame()

        matched_product = partial_match.iloc[0]

    product_id = matched_product["product_id"]
    selected_product = matched_product["product_name"]

    recommendations = recommendation_rules[
        (recommendation_rules["product_a_id"] == product_id)
        & (recommendation_rules["cooccurrence_count"] >= min_cooccurrence)
        & (recommendation_rules["lift"] >= min_lift)
        & (recommendation_rules["confidence"] >= min_confidence)
    ].copy()

    if recommendations.empty:
        print(f"Produto selecionado: {selected_product}")
        print("Nenhuma recomendação encontrada com os filtros atuais.")
        return pd.DataFrame()

    recommendations = (
        recommendations
        .sort_values(by=sort_by, ascending=False)
        .head(top_n)
    )

    print(f"Produto selecionado: {selected_product}")

    return recommendations[
        [
            "product_b",
            "cooccurrence_count",
            "confidence",
            "lift",
            "recommendation_score"
        ]
    ]

# 🧪 Teste da Função de Recomendação

Nesta etapa serão testadas recomendações para produtos populares da base.

O objetivo é verificar se as recomendações geradas fazem sentido do ponto de vista de associação entre produtos.

In [12]:
recommend_products(
    product_name="Organic Strawberries",
    top_n=10,
    min_cooccurrence=50,
    min_lift=1.2,
    min_confidence=0.01,
    sort_by="recommendation_score"
)

Produto selecionado: Organic Strawberries


,product_b,cooccurrence_count,confidence,lift,recommendation_score
8715,Bag of Organic Bananas,924,0.224272,1.868621,2.862223
466,Organic Hass Avocado,656,0.159223,2.400110,2.479291
1171,Organic Raspberries,519,0.125971,2.959842,2.331764
632,Banana,841,0.204126,1.407185,1.934808
1081,Organic Baby Spinach,568,0.137864,1.893737,1.656250
405,Organic Blueberries,350,0.084951,2.722803,1.355635
607,Organic Whole Milk,397,0.096359,2.245089,1.295079
1613,Organic Cucumber,277,0.067233,2.661639,1.007063
513,Organic Whole String Cheese,228,0.055340,2.988111,0.898528
240,Organic Grape Tomatoes,261,0.063350,2.427185,0.856194


In [13]:
recommend_products(
    product_name="Organic Whole Milk",
    top_n=10,
    min_cooccurrence=50,
    min_lift=1.2,
    min_confidence=0.01,
    sort_by="recommendation_score"
)

Produto selecionado: Organic Whole Milk


,product_b,cooccurrence_count,confidence,lift,recommendation_score
9294,Organic Strawberries,397,0.184995,2.245089,2.486359
9393,Banana,496,0.231128,1.593325,2.286384
9286,Bag of Organic Bananas,404,0.188257,1.568549,1.772892
10833,Organic Whole String Cheese,140,0.065238,3.522551,1.137240
171,Organic Hass Avocado,236,0.109972,1.657703,0.996833
9653,Organic Baby Spinach,244,0.113700,1.561812,0.976902
2637,Organic Avocado,202,0.094129,1.732216,0.866324
10531,Whole Milk Plain Yogurt,66,0.030755,6.078042,0.785981
2109,Organic Raspberries,171,0.079683,1.872254,0.767940
9301,Organic Garlic,127,0.059180,1.752958,0.503349


# 💡 Observação

Após a criação do `recommendation_score`, os resultados passaram a equilibrar melhor três aspectos importantes da recomendação: força da associação, frequência de coocorrência e probabilidade condicional.

Para o produto `Organic Strawberries`, as recomendações geradas incluem principalmente bananas, frutas orgânicas, abacate, espinafre e leite integral orgânico, indicando um padrão consistente de produtos frescos e saudáveis frequentemente comprados em conjunto.

Para o produto `Organic Whole Milk`, as recomendações também apresentaram coerência de cesta, incluindo frutas, iogurtes, queijos e vegetais orgânicos.

Embora a métrica `confidence` possa parecer baixa em alguns casos, esse comportamento é esperado em bases de supermercado com grande variedade de produtos. Por isso, o ranking final utiliza um score combinado entre `lift`, `confidence` e `cooccurrence_count`, reduzindo o viés de produtos muito populares e evitando recomendações baseadas em pares raros.

De forma geral, os resultados indicam que a abordagem de Market Basket Analysis é adequada como baseline interpretável para recomendação produto → produto.

In [14]:
# =========================================
# 💾 EXPORTAÇÃO DAS REGRAS DE RECOMENDAÇÃO
# =========================================

OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

recommendation_rules.to_parquet(
    OUTPUT_PATH / "market_basket_rules.parquet",
    index=False
)

print("✅ Regras de recomendação salvas com sucesso!")

✅ Regras de recomendação salvas com sucesso!


# 📌 Conclusão

Neste notebook foi construída uma primeira abordagem de recomendação produto → produto utilizando Market Basket Analysis.

A partir de uma amostra de pedidos, foram identificados pares de produtos frequentemente comprados em conjunto e calculadas métricas de associação, como coocorrência, confidence e lift.

A métrica principal adotada foi o `lift`, pois ela permite identificar associações proporcionalmente mais fortes do que o esperado pela popularidade individual dos produtos. Além disso, foi criado um `recommendation_score` combinando lift, confidence e volume de coocorrência, tornando o ranking mais equilibrado.

Os resultados obtidos demonstraram recomendações coerentes para produtos populares, especialmente em categorias de frutas, vegetais, orgânicos, laticínios e itens frescos.

Essa abordagem será utilizada como baseline interpretável para comparação com modelos mais avançados de recomendação, como modelos baseados em similaridade e embeddings treinados com PyTorch.